# 00 - Raw Data Understanding



In [3]:
# pathlib helps us work with file paths in a clean and readable way.
from pathlib import Path

# json is needed because the raw files are JSON Lines files.
# JSON Lines means: one JSON object per line.
import json

# pandas is used to display tables and inspect samples.
import pandas as pd

# Show more columns in notebook outputs so we can inspect raw schemas comfortably.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

# Detect the project root in a simple way.
# In some tools, the notebook runs from the project root.
# In other tools, the notebook runs from the notebooks/ folder.
current_working_directory_path = Path.cwd()

# If data/raw exists in the current working directory, we are already at the project root.
if (current_working_directory_path / "data/raw").exists():
    project_root_path = current_working_directory_path

# Otherwise, assume the notebook is running from notebooks/ and move one folder up.
else:
    project_root_path = current_working_directory_path.parent

# This is the root folder that contains all extracted raw data.
raw_data_root_path = project_root_path / "data/raw"

# These are the three raw collections used by the project.
# Orders and delivery requests are business data.
# Performance logs are technical/backend data.
raw_orders_folder_path = raw_data_root_path / "orders"
raw_delivery_requests_folder_path = raw_data_root_path / "delivery_requests"
raw_performance_logs_folder_path = raw_data_root_path / "performance_logs"

# We only load a small sample from each file.
# This keeps the notebook fast and easy to use.
sample_row_limit_per_file = 1_000

In [4]:
def list_jsonl_files(folder_path):
    """Return all JSONL files inside one folder, sorted by file name."""
    
    # If the folder does not exist, return an empty list instead of crashing.
    if not folder_path.exists():
        return []
    
    # The raw extraction files use the .jsonl extension.
    return sorted(folder_path.glob("*.jsonl"))


def count_non_empty_jsonl_lines(file_path):
    """Count non-empty lines in a JSONL file.
    
    In JSONL files, one valid line usually means one raw document.
    This is not a deep validation. It is only a fast row-count check.
    """
    
    # Open in binary mode because it is slightly faster for simple line counting.
    with file_path.open("rb") as raw_file:
        return sum(1 for line in raw_file if line.strip())


def load_jsonl_sample(file_path, row_limit=1_000):
    """Load only the first N valid JSON records from one JSONL file."""
    
    # Store parsed JSON documents here before converting them to a dataframe.
    parsed_records = []
    
    # Open the file as text because we need to parse JSON strings.
    with file_path.open("r", encoding="utf-8") as raw_file:
        for raw_line in raw_file:
            
            # Stop as soon as we have enough rows for understanding.
            if len(parsed_records) >= row_limit:
                break
            
            # Remove extra spaces/newlines around the JSON string.
            clean_line = raw_line.strip()
            
            # Skip empty lines because they are not useful data records.
            if not clean_line:
                continue
            
            # Try to parse the JSON record.
            # If one line is broken, skip it for now and investigate later if needed.
            try:
                parsed_records.append(json.loads(clean_line))
            except json.JSONDecodeError:
                continue
    
    # Convert the sample records into a dataframe for inspection.
    return pd.DataFrame(parsed_records)


def get_nested_value(raw_record, dotted_field_name):
    """Extract a nested value from a raw JSON dictionary using dot notation.
    
    Example:
    dotted_field_name = 'destination.address.city'
    """
    
    # Start from the full raw record.
    current_value = raw_record
    
    # Move one level deeper for each part of the dotted field name.
    for field_part in dotted_field_name.split("."):
        
        # If the current value is not a dictionary, the nested path does not exist.
        if not isinstance(current_value, dict):
            return None
        
        # Move to the next nested value.
        current_value = current_value.get(field_part)
    
    return current_value


def extract_nested_field_as_series(dataframe, dotted_field_name):
    """Extract a normal or nested field from a dataframe as a pandas Series."""
    
    # If the field already exists as a normal dataframe column, return it directly.
    if dotted_field_name in dataframe.columns:
        return dataframe[dotted_field_name]
    
    # The first part is the top-level JSON column, for example 'destination'.
    top_level_field_name = dotted_field_name.split(".")[0]
    
    # If the top-level field does not exist, return an empty-like series of None values.
    if top_level_field_name not in dataframe.columns:
        return pd.Series([None] * len(dataframe), index=dataframe.index)
    
    # Extract the nested value from each raw dictionary.
    return dataframe.apply(lambda row: get_nested_value(row.to_dict(), dotted_field_name), axis=1)


def extract_mongo_date_value(raw_date_value):
    """Convert MongoDB exported date values into plain values pandas can parse."""
    
    # mongoexport commonly writes dates as dictionaries like {'$date': '2025-...'}.
    if isinstance(raw_date_value, dict) and "$date" in raw_date_value:
        return raw_date_value["$date"]
    
    # If it is already a string or null, return it as-is.
    return raw_date_value


def missing_rate_for_field(dataframe, dotted_field_name):
    """Calculate the missing rate for a normal or nested field."""
    
    # Extract the field values first.
    field_values = extract_nested_field_as_series(dataframe, dotted_field_name)
    
    # Treat common string placeholders as missing values.
    missing_text_values = {"", "none", "null", "nan"}
    
    def value_is_missing(value):
        # None is missing.
        if value is None:
            return True
        
        # Empty dictionaries/lists are missing.
        if isinstance(value, (dict, list, tuple, set)):
            return len(value) == 0
        
        # pandas can detect NaN values.
        try:
            if pd.isna(value):
                return True
        except TypeError:
            pass
        
        # Text placeholders are also treated as missing.
        return str(value).strip().lower() in missing_text_values
    
    return field_values.map(value_is_missing).mean()


def show_status_distribution(dataframe, field_name):
    """Show the distribution of a status-like field."""
    
    # Extract values as lowercase text so small formatting differences do not create fake categories.
    status_values = extract_nested_field_as_series(dataframe, field_name).dropna().astype(str).str.lower().str.strip()
    
    # Display the most common values.
    return status_values.value_counts(dropna=False).to_frame("row_count")

## 3. Raw File Inventory

This section answers: **What raw files do we have and how large are they?**

This is the first extraction sanity check.

In [5]:
# Store all raw collection folders in one clearly named dictionary.
raw_collection_folder_paths = {
    "orders": raw_orders_folder_path,
    "delivery_requests": raw_delivery_requests_folder_path,
    "performance_logs": raw_performance_logs_folder_path,
}

# Each row in this list will become one row in the inventory dataframe.
raw_file_inventory_rows = []

# Loop over each collection and its folder.
for collection_name, collection_folder_path in raw_collection_folder_paths.items():
    
    # Find all JSONL files for this collection.
    raw_jsonl_file_paths = list_jsonl_files(collection_folder_path)
    
    # Add one inventory row per file.
    for raw_jsonl_file_path in raw_jsonl_file_paths:
        raw_file_inventory_rows.append({
            "collection_name": collection_name,
            "file_name": raw_jsonl_file_path.name,
            "file_path": str(raw_jsonl_file_path),
            "file_size_mb": round(raw_jsonl_file_path.stat().st_size / 1024 / 1024, 2),
            "row_count": count_non_empty_jsonl_lines(raw_jsonl_file_path),
        })

# Convert the inventory list into a dataframe for display.
raw_file_inventory = pd.DataFrame(raw_file_inventory_rows)

# Display the raw file inventory.
raw_file_inventory

,collection_name,file_name,file_path,file_size_mb,row_count
0,orders,orders_2022-01-01_2022-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\orders\orders_2022-01-01_2022-12-31.jsonl,777.08,934216
1,orders,orders_2023-01-01_2023-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\orders\orders_2023-01-01_2023-12-31.jsonl,887.43,1068649
2,orders,orders_2024-01-01_2024-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\orders\orders_2024-01-01_2024-12-31.jsonl,1014.52,1205610
3,orders,orders_2025-01-01_2025-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\orders\orders_2025-01-01_2025-12-31.jsonl,1031.73,1244725
4,orders,orders_2026-01-01_2026-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\orders\orders_2026-01-01_2026-12-31.jsonl,232.99,289331
5,delivery_requests,delivery_requests_2022-01-01_2022-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\delivery_requests\delivery_requests_2022-01-01_2022-12-31.jsonl,610.12,3177642
6,delivery_requests,delivery_requests_2023-01-01_2023-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\delivery_requests\delivery_requests_2023-01-01_2023-12-31.jsonl,554.04,2885259
7,delivery_requests,delivery_requests_2024-01-01_2024-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\delivery_requests\delivery_requests_2024-01-01_2024-12-31.jsonl,756.61,3941437
8,delivery_requests,delivery_requests_2025-01-01_2025-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\delivery_requests\delivery_requests_2025-01-01_2025-12-31.jsonl,992.98,5174043
9,delivery_requests,delivery_requests_2026-01-01_2026-12-31.jsonl,c:\stage_armada\business_load_prediction\data\raw\delivery_requests\delivery_requests_2026-01-01_2026-12-31.jsonl,217.88,1135562


## 4. Load Small Samples

This section loads only a small sample from each raw file.

Important: this is **not** the full dataset. It is only for understanding the structure quickly.

In [6]:
# This dictionary will store one sampled dataframe per collection.
raw_collection_samples = {}

# Loop over each collection.
for collection_name, collection_folder_path in raw_collection_folder_paths.items():
    
    # This list stores all file samples for the current collection.
    collection_sample_dataframes = []
    
    # Load a small sample from each file in this collection.
    for raw_jsonl_file_path in list_jsonl_files(collection_folder_path):
        
        # Load only the first N rows from this file.
        file_sample_dataframe = load_jsonl_sample(
            raw_jsonl_file_path,
            row_limit=sample_row_limit_per_file,
        )
        
        # Add metadata so we know where each sampled row came from.
        file_sample_dataframe["source_file_name"] = raw_jsonl_file_path.name
        file_sample_dataframe["source_collection_name"] = collection_name
        
        # Save this file sample for later concatenation.
        collection_sample_dataframes.append(file_sample_dataframe)
    
    # Combine all file samples for this collection.
    if collection_sample_dataframes:
        raw_collection_samples[collection_name] = pd.concat(collection_sample_dataframes, ignore_index=True)
    else:
        raw_collection_samples[collection_name] = pd.DataFrame()

# Give simple names to the three sampled datasets.
raw_orders_sample = raw_collection_samples["orders"]
raw_delivery_requests_sample = raw_collection_samples["delivery_requests"]
raw_performance_logs_sample = raw_collection_samples["performance_logs"]

# Display sample sizes so we know how much data we loaded into memory.
pd.DataFrame([
    {"collection_name": "orders", "sample_rows": len(raw_orders_sample), "sample_columns": len(raw_orders_sample.columns)},
    {"collection_name": "delivery_requests", "sample_rows": len(raw_delivery_requests_sample), "sample_columns": len(raw_delivery_requests_sample.columns)},
    {"collection_name": "performance_logs", "sample_rows": len(raw_performance_logs_sample), "sample_columns": len(raw_performance_logs_sample.columns)},
])

,collection_name,sample_rows,sample_columns
0,orders,5000,14
1,delivery_requests,5000,7
2,performance_logs,2000,8


## 5. Orders Understanding

Orders are the main business event. This dataset is the foundation for area/hour traffic prediction.

In [7]:
# Show the first few sampled raw order records.
# We are checking the structure, not doing cleaning yet.
raw_orders_sample.head()

,_id,amount,orderStatus,driverSearchTrials,country,deleted,destination,platformName,merchant,branch,deliveryFee,createdAt,source_file_name,source_collection_name
0,{'$oid': '61cf99c3dbaf06002f414ff6'},3.30,failed,31,Bahrain,False,"{'phone': '97336221841', 'name': 'Shazly Abdulkrem', 'address': {'firstLine': 'AMWAJ, Block 257, St 5722, 1722', 'ar...",mcd,{'$oid': '613879c2d0207f003116ebd1'},{'$oid': '6199024369696900359cea75'},1.0,{'$date': '2022-01-01T00:01:08.26Z'},orders_2022-01-01_2022-12-31.jsonl,orders
1,{'$oid': '61cf99c661c53b002a330366'},3.70,failed,31,Bahrain,False,"{'phone': '97336851471', 'name': 'Ismail Echair', 'address': {'firstLine': 'AMWAJ, Block 257, Road No 5715, 1830', '...",mcd,{'$oid': '613879c2d0207f003116ebd1'},{'$oid': '6199024369696900359cea75'},1.0,{'$date': '2022-01-01T00:01:11.1Z'},orders_2022-01-01_2022-12-31.jsonl,orders
2,{'$oid': '61cf99c90517910039429a28'},5.60,failed,31,Bahrain,False,"{'phone': '97335526800', 'name': 'Faisal Dabbagh', 'address': {'firstLine': 'AMWAJ, Block 257, St 5705, No. 3804', '...",mcd,{'$oid': '613879c2d0207f003116ebd1'},{'$oid': '6199024369696900359cea75'},1.0,{'$date': '2022-01-01T00:01:13.883Z'},orders_2022-01-01_2022-12-31.jsonl,orders
3,{'$oid': '61cf99cbfeb1c600244be953'},3.30,failed,31,Bahrain,False,"{'phone': '97333435302', 'name': 'نوره الدوسري', 'address': {'firstLine': 'HIDD, Block 106, Road No 601, No. 7', 'ar...",mcd,{'$oid': '613879c2d0207f003116ebd1'},{'$oid': '6199024369696900359cea75'},1.0,{'$date': '2022-01-01T00:01:16.011Z'},orders_2022-01-01_2022-12-31.jsonl,orders
4,{'$oid': '61cf9a3548d905002abfad9a'},2.25,completed,20,Bahrain,False,"{'phone': '97333122088', 'name': 'عبدالعزيز الفارس', 'address': {'firstLine': 'LAWZI, Block 1016, St 1610, 857', 'ar...",mcd,{'$oid': '613879c2d0207f003116ebd1'},{'$oid': '6152f69fbf68f4002da0c63c'},1.0,{'$date': '2022-01-01T00:03:02.11Z'},orders_2022-01-01_2022-12-31.jsonl,orders


In [8]:
# Show the top-level columns available in raw orders.
# This tells us whether extraction included the fields needed by clean_orders.py.
pd.DataFrame({"raw_orders_column_name": raw_orders_sample.columns})

,raw_orders_column_name
0,_id
1,amount
2,orderStatus
3,driverSearchTrials
4,country
5,deleted
6,destination
7,platformName
8,merchant
9,branch


In [9]:
# Convert sampled createdAt values to pandas datetimes.
# This checks whether the raw date format can be parsed.
raw_orders_created_at = pd.to_datetime(
    raw_orders_sample["createdAt"].map(extract_mongo_date_value),
    errors="coerce",
    utc=True,
)

# Show the sampled date range.
pd.DataFrame([{
    "sample_min_createdAt": raw_orders_created_at.min(),
    "sample_max_createdAt": raw_orders_created_at.max(),
    "sample_missing_createdAt_rate": raw_orders_created_at.isna().mean(),
}])

,sample_min_createdAt,sample_max_createdAt,sample_missing_createdAt_rate
0,2022-01-01 00:01:08.260000+00:00,2026-01-01 10:47:34.685000+00:00,0.0014


In [10]:
# Check order status values.
# These values will later become completed/canceled/failed/other order counts.
show_status_distribution(raw_orders_sample, "orderStatus")

,row_count
orderStatus,
completed,4521
canceled,424
failed,55


In [11]:
# Check missing rates for important order fields.
# These are sample-based checks only.
order_important_field_names = [
    "_id",
    "createdAt",
    "orderStatus",
    "destination",
    "destination.address.city",
    "destination.address.area",
    "destination.address.street",
    "destination.address.location",
    "destination.address.location.latitude",
    "destination.address.location.longitude",
    "amount",
    "deliveryFee",
    "driverSearchTrials",
    "country",
    "platformName",
]

order_missing_rate_rows = []

for field_name in order_important_field_names:
    order_missing_rate_rows.append({
        "field_name": field_name,
        "sample_missing_rate": missing_rate_for_field(raw_orders_sample, field_name),
    })

pd.DataFrame(order_missing_rate_rows).sort_values("sample_missing_rate", ascending=False)

,field_name,sample_missing_rate
14,platformName,0.5518
6,destination.address.street,0.0432
4,destination.address.city,0.0014
5,destination.address.area,0.0014
2,orderStatus,0.0000
0,_id,0.0000
1,createdAt,0.0000
3,destination,0.0000
7,destination.address.location,0.0000
9,destination.address.location.longitude,0.0000


In [12]:
# Inspect the raw address fields together.
# This is important because area normalization is one of the most sensitive parts of the project.
order_address_review = pd.DataFrame({
    "city": extract_nested_field_as_series(raw_orders_sample, "destination.address.city"),
    "area": extract_nested_field_as_series(raw_orders_sample, "destination.address.area"),
    "street": extract_nested_field_as_series(raw_orders_sample, "destination.address.street"),
    "location": extract_nested_field_as_series(raw_orders_sample, "destination.address.location"),
    "source_file_name": raw_orders_sample["source_file_name"],
})

# Display a sample of address values.
order_address_review.head(20)

,city,area,street,location,source_file_name
0,AMWAJ,AMWAJ,St 5722,"{'latitude': 26.2901508, 'longitude': 50.6599176}",orders_2022-01-01_2022-12-31.jsonl
1,AMWAJ,AMWAJ,Road No 5715,"{'latitude': 26.2856067, 'longitude': 50.6654876}",orders_2022-01-01_2022-12-31.jsonl
2,AMWAJ,AMWAJ,St 5705,"{'latitude': 26.291111, 'longitude': 50.665278}",orders_2022-01-01_2022-12-31.jsonl
3,HIDD,HIDD,Road No 601,"{'latitude': 26.2463443, 'longitude': 50.6566511}",orders_2022-01-01_2022-12-31.jsonl
4,LAWZI,LAWZI,St 1610,"{'latitude': 26.1239003, 'longitude': 50.4851036}",orders_2022-01-01_2022-12-31.jsonl
5,LAWZI,LAWZI,St 2019,"{'latitude': 26.1239003, 'longitude': 50.4851036}",orders_2022-01-01_2022-12-31.jsonl
6,Nahda,Nahda,St 303,"{'latitude': 29.3076, 'longitude': 47.8578}",orders_2022-01-01_2022-12-31.jsonl
7,DAMISTAN,DAMISTAN,St 2254,"{'latitude': 26.1237836, 'longitude': 50.4695879}",orders_2022-01-01_2022-12-31.jsonl
8,HAMAD TOWN,HAMAD TOWN,Road No 611,"{'latitude': 26.1270093, 'longitude': 50.5005208}",orders_2022-01-01_2022-12-31.jsonl
9,Sabah Al-Salem,Sabah Al-Salem,St 2 Lane7,"{'latitude': 29.245014, 'longitude': 48.077112}",orders_2022-01-01_2022-12-31.jsonl


In [13]:
# Numeric sanity check for order monetary and search-trial fields.
# Extreme values here may become outliers later.
order_numeric_field_names = ["amount", "deliveryFee", "driverSearchTrials"]

order_numeric_summary_rows = []

for numeric_field_name in order_numeric_field_names:
    numeric_values = pd.to_numeric(raw_orders_sample[numeric_field_name], errors="coerce")
    order_numeric_summary_rows.append({
        "field_name": numeric_field_name,
        "sample_non_null_count": numeric_values.notna().sum(),
        "sample_min": numeric_values.min(),
        "sample_p50": numeric_values.quantile(0.50),
        "sample_p95": numeric_values.quantile(0.95),
        "sample_max": numeric_values.max(),
    })

pd.DataFrame(order_numeric_summary_rows)

,field_name,sample_non_null_count,sample_min,sample_p50,sample_p95,sample_max
0,amount,5000,0.00,9.15,29.405,326079.00
1,deliveryFee,5000,0.75,1.50,3.000,17.25
2,driverSearchTrials,5000,0.00,2.00,20.000,61.00


## 6. Delivery Requests Understanding

Delivery requests describe how many drivers were requested and what happened to those requests. This is important for pressure signals.

In [14]:
# Show the first few sampled raw delivery request records.
raw_delivery_requests_sample.head()

,_id,status,order,driver,createdAt,source_file_name,source_collection_name
0,{'$oid': '61cf9986dce4cd00164605a2'},pending,{'$oid': '61cf993529f71c0024f0be02'},{'$oid': '614b20d9ca1ef5002f8c9c39'},{'$date': '2022-01-01T00:00:06.051Z'},delivery_requests_2022-01-01_2022-12-31.jsonl,delivery_requests
1,{'$oid': '61cf99b3dce4cd00164612ea'},pending,{'$oid': '61cf95a80e60620036da2f9d'},{'$oid': '619903cbc2d4dd0036e5e66a'},{'$date': '2022-01-01T00:00:51.018Z'},delivery_requests_2022-01-01_2022-12-31.jsonl,delivery_requests
2,{'$oid': '61cf99b3dce4cd00164612f7'},accepted,{'$oid': '61cf992cdbaf06002f414f51'},{'$oid': '613defabeed6f40038b76f68'},{'$date': '2022-01-01T00:00:51.376Z'},delivery_requests_2022-01-01_2022-12-31.jsonl,delivery_requests
3,{'$oid': '61cf99c2dce4cd0016461352'},pending,{'$oid': '61cf993529f71c0024f0be02'},{'$oid': '61c17f95dcc510002f2ee8a5'},{'$date': '2022-01-01T00:01:06.109Z'},delivery_requests_2022-01-01_2022-12-31.jsonl,delivery_requests
4,{'$oid': '61cf99d1dce4cd0016461372'},pending,{'$oid': '61cf99c3dbaf06002f414ff6'},{'$oid': '61c41caf6488c3002ab5b4fe'},{'$date': '2022-01-01T00:01:21.014Z'},delivery_requests_2022-01-01_2022-12-31.jsonl,delivery_requests


In [15]:
# Show top-level columns available in raw delivery requests.
pd.DataFrame({"raw_delivery_request_column_name": raw_delivery_requests_sample.columns})

,raw_delivery_request_column_name
0,_id
1,status
2,order
3,driver
4,createdAt
5,source_file_name
6,source_collection_name


In [16]:
# Convert sampled delivery request createdAt values to pandas datetimes.
raw_delivery_requests_created_at = pd.to_datetime(
    raw_delivery_requests_sample["createdAt"].map(extract_mongo_date_value),
    errors="coerce",
    utc=True,
)

# Show sampled delivery request date range.
pd.DataFrame([{
    "sample_min_createdAt": raw_delivery_requests_created_at.min(),
    "sample_max_createdAt": raw_delivery_requests_created_at.max(),
    "sample_missing_createdAt_rate": raw_delivery_requests_created_at.isna().mean(),
}])

,sample_min_createdAt,sample_max_createdAt,sample_missing_createdAt_rate
0,2022-01-01 00:00:06.051000+00:00,2026-01-01 07:18:07.521000+00:00,0.001


In [17]:
# Check delivery request status values.
# These values later become accepted/rejected/ignored/pending request counts.
show_status_distribution(raw_delivery_requests_sample, "status")

,row_count
status,
pending,3029
accepted,1606
rejected,365


In [18]:
# Check missing rates for important delivery request fields.
# We do not join delivery requests to orders in this notebook.
delivery_request_important_field_names = [
    "_id",
    "createdAt",
    "order",
    "driver",
    "status",
]

delivery_request_missing_rate_rows = []

for field_name in delivery_request_important_field_names:
    delivery_request_missing_rate_rows.append({
        "field_name": field_name,
        "sample_missing_rate": missing_rate_for_field(raw_delivery_requests_sample, field_name),
    })

pd.DataFrame(delivery_request_missing_rate_rows).sort_values("sample_missing_rate", ascending=False)

,field_name,sample_missing_rate
0,_id,0.0
1,createdAt,0.0
2,order,0.0
3,driver,0.0
4,status,0.0


## 7. Performance Logs Understanding

Performance logs describe backend technical pressure: endpoint request counts, response time, and errors.

In [19]:
# Show the first few sampled raw performance log records.
raw_performance_logs_sample.head()

,_id,endpoint,method,statusCode,responseTime,createdAt,source_file_name,source_collection_name
0,{'$oid': '6955b8815c8199004930d6ea'},/drivers/orders/details,GET,200,2776,{'$date': '2025-12-31T23:57:53.536Z'},performance_logs_2025-01-01_2025-12-31.jsonl,performance_logs
1,{'$oid': '6955b4190d243f0039aa8aca'},/drivers/orders/details,GET,200,2864,{'$date': '2025-12-31T23:39:05.462Z'},performance_logs_2025-01-01_2025-12-31.jsonl,performance_logs
2,{'$oid': '6955b32dfe92b80040290108'},/drivers/orders/details,GET,200,1527,{'$date': '2025-12-31T23:35:09.668Z'},performance_logs_2025-01-01_2025-12-31.jsonl,performance_logs
3,{'$oid': '6955b0d1cf6eef004743b8cb'},/drivers/orders/details,GET,200,1510,{'$date': '2025-12-31T23:25:05.558Z'},performance_logs_2025-01-01_2025-12-31.jsonl,performance_logs
4,{'$oid': '6955acab0d243f0039aa4234'},/drivers/orders/details,GET,200,5102,{'$date': '2025-12-31T23:07:23.92Z'},performance_logs_2025-01-01_2025-12-31.jsonl,performance_logs


In [20]:
# Show top-level columns available in raw performance logs.
pd.DataFrame({"raw_performance_log_column_name": raw_performance_logs_sample.columns})

,raw_performance_log_column_name
0,_id
1,endpoint
2,method
3,statusCode
4,responseTime
5,createdAt
6,source_file_name
7,source_collection_name


In [21]:
# Convert sampled performance log createdAt values to pandas datetimes.
raw_performance_logs_created_at = pd.to_datetime(
    raw_performance_logs_sample["createdAt"].map(extract_mongo_date_value),
    errors="coerce",
    utc=True,
)

# Show sampled performance log date range.
# Reminder: this is based on samples, not exact full-file min/max.
pd.DataFrame([{
    "sample_min_createdAt": raw_performance_logs_created_at.min(),
    "sample_max_createdAt": raw_performance_logs_created_at.max(),
    "sample_missing_createdAt_rate": raw_performance_logs_created_at.isna().mean(),
}])

,sample_min_createdAt,sample_max_createdAt,sample_missing_createdAt_rate
0,2025-12-31 19:12:12.383000+00:00,2026-05-19 14:02:42.781000+00:00,0.0005


In [22]:
# Count sampled performance log rows by source file.
# This confirms that technical logs exist only for extracted years with files.
raw_performance_logs_sample["source_file_name"].value_counts().to_frame("sample_rows")

,sample_rows
source_file_name,
performance_logs_2025-01-01_2025-12-31.jsonl,1000
performance_logs_2026-01-01_2026-12-31.jsonl,1000


In [23]:
# Check common HTTP methods in the sampled technical logs.
show_status_distribution(raw_performance_logs_sample, "method")

,row_count
method,
get,1472
post,423
put,105


In [24]:
# Check common status codes in the sampled technical logs.
show_status_distribution(raw_performance_logs_sample, "statusCode")

,row_count
statusCode,
200,1487
304,470
204,36
403,4
400,2
404,1


In [25]:
# Check missing rates for important performance log fields.
performance_log_important_field_names = [
    "_id",
    "createdAt",
    "endpoint",
    "method",
    "statusCode",
    "responseTime",
]

performance_log_missing_rate_rows = []

for field_name in performance_log_important_field_names:
    performance_log_missing_rate_rows.append({
        "field_name": field_name,
        "sample_missing_rate": missing_rate_for_field(raw_performance_logs_sample, field_name),
    })

pd.DataFrame(performance_log_missing_rate_rows).sort_values("sample_missing_rate", ascending=False)

,field_name,sample_missing_rate
0,_id,0.0
1,createdAt,0.0
2,endpoint,0.0
3,method,0.0
4,statusCode,0.0
5,responseTime,0.0


In [26]:
# Numeric sanity check for response time.
# Extreme values here may indicate slow endpoints or measurement issues.
performance_response_time_values = pd.to_numeric(raw_performance_logs_sample["responseTime"], errors="coerce")

pd.DataFrame([{
    "sample_non_null_count": performance_response_time_values.notna().sum(),
    "sample_min": performance_response_time_values.min(),
    "sample_p50": performance_response_time_values.quantile(0.50),
    "sample_p95": performance_response_time_values.quantile(0.95),
    "sample_p99": performance_response_time_values.quantile(0.99),
    "sample_max": performance_response_time_values.max(),
}])

,sample_non_null_count,sample_min,sample_p50,sample_p95,sample_p99,sample_max
0,2000,10,1004.0,4529.1,11434.15,50820
